In [0]:
# =============================================================================
# 0. Configurações e funções utilitárias
# =============================================================================
from pyspark.sql import functions as F

CATALOG = "cinedata_analytics"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD} COMMENT 'Camada Gold - Star Schema e tabelas de consumo'")

def ler_silver(tabela):
    return spark.table(f"{SILVER}.{tabela}")

def ler_gold(tabela):
    return spark.table(f"{GOLD}.{tabela}")

def salvar_gold(df, tabela):
    """Gold reconstruída a cada execução (overwrite) a partir da Silver, de forma idempotente."""
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.{tabela}"))
    print(f"OK  {GOLD}.{tabela}: {ler_gold(tabela).count():,} linhas")

def gerar_sk(*colunas):
    """Surrogate key BIGINT derivada de sha2 (hash) da chave natural.
    - sha2(chave, 256) gera um hash hexadecimal determinístico: a mesma chave natural
      SEMPRE produz a mesma SK, em qualquer execução do Job. Com row_number() ou
      monotonically_increasing_id(), a SK de um filme mudaria quando entrassem filmes novos
      ou mudasse o particionamento, quebrando as FKs de quem consome a Gold.
    - Os primeiros 15 caracteres hex (60 bits) são convertidos para decimal (conv) e depois
      para BIGINT (positivo, cabe em 63 bits). Com ~1 milhão de chaves, a chance de colisão
      é da ordem de 1 em 1 bilhão, e mesmo assim cada dimensão é validada abaixo.
    - concat_ws com separador '||' evita ambiguidade em chaves compostas (nome + tipo)."""
    chave = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in colunas])
    return F.conv(F.substring(F.sha2(chave, 256), 1, 15), 16, 10).cast("BIGINT")

def validar_unicidade(df, coluna, nome):
    """Garante que a coluna é única (PK/grão). Se não for, interrompe o pipeline:
    melhor falhar do que publicar uma fato ou dimensão com linhas duplicadas."""
    total, distintos = df.count(), df.select(coluna).distinct().count()
    if total != distintos:
        raise ValueError(f"{nome}: {total - distintos} valores duplicados em '{coluna}'")
    print(f"Unicidade OK  {nome}.{coluna}  ({total:,} linhas)")

In [0]:
# =============================================================================
# 1. gold.dim_movies
# =============================================================================
# Contém TODOS os filmes (lançados ou não): a dimensão descreve o catálogo inteiro.
# O recorte "somente lançados" é aplicado na tabela fato (regra do grão).
dim_movies = (ler_silver("tb_info_filmes")
    .select(
        gerar_sk("id_filme").alias("sk_movie_id"),
        F.col("id_filme").cast("STRING").alias("id_filme"),
        F.col("titulo").cast("STRING"),
        F.col("data_lancamento").cast("DATE"),
        F.col("ano_lancamento").cast("INT"),
        F.col("duracao_minutos").cast("INT"),
        F.col("idioma_original").cast("STRING"),
        F.col("status_filme").cast("STRING"),
        F.col("sinopse").cast("STRING"),
    ))

validar_unicidade(dim_movies, "id_filme", "dim_movies")
validar_unicidade(dim_movies, "sk_movie_id", "dim_movies")
salvar_gold(dim_movies, "dim_movies")
display(ler_gold("dim_movies").limit(10))

Unicidade OK  dim_movies.id_filme  (97,611 linhas)
Unicidade OK  dim_movies.sk_movie_id  (97,611 linhas)
OK  cinedata_analytics.gold.dim_movies: 97,611 linhas


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
7439950412128225,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
752552491917266933,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
195172450439143704,1000007,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
592966999998764912,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
266212661144708360,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,null,es,Lançado,null
538678607616135496,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
1039669485397040762,1000079,Pourquoi tu souris ?,2024-06-26,2024,null,fr,Em Produção,null
188846793746616210,1000081,Sentinelle,2023-08-27,2023,98,fr,Lançado,"François Sentinelle has two lives. By day, he is the most famous cop of Réunion Island, known for his tough methods and flowery shirts, pursuing criminals in his famous yellow defender. But the rest of the time, Sentinelle is also a charming singer."
61439698267836763,1000091,Amy Miller: Ham Mouth,2022-03-24,2022,35,en,Lançado,"Amy Miller reflects on her recent breakup, shares her love of baths and reveals the 40-year-old shit she does."
958883485799444975,1000094,Le clan,2023-01-18,2023,90,fr,Lançado,"Fred, Achille, Max and Belette form a good-for-nothing gang of crooks. Dismally failing their last raid, they decide to get back in the game, kidnapping Sophie Marceau."


In [0]:
# =============================================================================
# 2. gold.dim_genres
# =============================================================================
# Catálogo único: distinct sobre o nome (a Silver já validou contra o domínio de gêneros).
dim_genres = (ler_silver("tb_generos")
    .select(F.col("nome_genero").cast("STRING"))
    .distinct()
    .withColumn("sk_genre_id", gerar_sk("nome_genero"))
    .select("sk_genre_id", "nome_genero"))

validar_unicidade(dim_genres, "sk_genre_id", "dim_genres")
salvar_gold(dim_genres, "dim_genres")
display(ler_gold("dim_genres").orderBy("nome_genero"))

Unicidade OK  dim_genres.sk_genre_id  (19 linhas)
OK  cinedata_analytics.gold.dim_genres: 19 linhas


sk_genre_id,nome_genero
454018119960296748,Action
694611403397729469,Adventure
134282196610852721,Animation
603232265097595755,Comedy
154830817088025809,Crime
305998218725573639,Documentary
983803378021321891,Drama
851979089117791105,Family
379011919105029965,Fantasy
65136580812306694,History


In [0]:
# =============================================================================
# 3. gold.dim_people  e  4. gold.dim_companies
# =============================================================================
# A Silver unifica pessoas e empresas; na Gold elas são separadas em duas dimensões.
# dim_people: a chave natural é (nome, tipo). A mesma pessoa como Ator e como Diretor gera
# 2 linhas, porque tipo_pessoa é atributo da dimensão, como pede o escopo.
dim_people = (ler_silver("tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").cast("STRING").alias("nome_pessoa"),
            F.col("tipo_entidade").cast("STRING").alias("tipo_pessoa"))
    .distinct()
    .withColumn("sk_person_id", gerar_sk("nome_pessoa", "tipo_pessoa"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa"))

dim_companies = (ler_silver("tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").cast("STRING").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", gerar_sk("nome_produtora"))
    .select("sk_company_id", "nome_produtora"))

validar_unicidade(dim_people, "sk_person_id", "dim_people")
validar_unicidade(dim_companies, "sk_company_id", "dim_companies")
salvar_gold(dim_people, "dim_people")
salvar_gold(dim_companies, "dim_companies")

display(ler_gold("dim_people").groupBy("tipo_pessoa").count())
display(ler_gold("dim_companies").limit(10))

Unicidade OK  dim_people.sk_person_id  (417,525 linhas)
Unicidade OK  dim_companies.sk_company_id  (45,741 linhas)
OK  cinedata_analytics.gold.dim_people: 417,525 linhas
OK  cinedata_analytics.gold.dim_companies: 45,741 linhas


tipo_pessoa,count
Ator,269069
Roteirista,84777
Diretor,63679


sk_company_id,nome_produtora
836048548435946989,Brad Krevoy Television
442869424831698596,VGTV
900827076717291344,Hombale Films
787852733631209813,Paranoi Future Cinema Lab
1015352378363462273,Universal Pictures
466050302049909128,Rohde-Dahl Filmproduktion
982600249813609012,and she has completely lost her memory. Liam
896779129233007484,Mahaateja Creations
107492036425845524,Rue Morgue Cinema
223793921638654,Essential Filmproduktion


In [0]:
# =============================================================================
# 5. gold.dim_reviews
# =============================================================================
# Transforma avaliações individuais em uma métrica resumida por filme (1 linha por filme).
# - qtd_avaliacoes_usuarios: conta TODAS as avaliações, inclusive as com nota NULL
#   (nota fora da escala), porque a avaliação existiu.
# - nota_media_usuarios: avg ignora NULLs, então notas inválidas não distorcem a média.
# - Inner join com dim_movies: só entram avaliações de filmes existentes no catálogo
#   (integridade referencial da FK sk_movie_id).
df_resumo_reviews = (ler_silver("tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("INT").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("DOUBLE").alias("nota_media_usuarios")))

dim_reviews = (df_resumo_reviews
    .join(ler_gold("dim_movies").select("id_filme", "sk_movie_id"), "id_filme", "inner")
    # SK da review: hash de "review||id_filme". O prefixo garante uma SK diferente da sk_movie_id,
    # mesmo sendo 1 resumo por filme (cada tabela tem seu próprio espaço de chaves).
    .withColumn("_chave_review", F.concat(F.lit("review||"), F.col("id_filme")))
    .withColumn("sk_review_id", gerar_sk("_chave_review"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"))

orfas = df_resumo_reviews.join(ler_gold("dim_movies"), "id_filme", "left_anti").count()
print(f"Filmes com avaliação mas fora do catálogo (descartados): {orfas}")

validar_unicidade(dim_reviews, "sk_movie_id", "dim_reviews")
salvar_gold(dim_reviews, "dim_reviews")
display(ler_gold("dim_reviews").orderBy(F.desc("qtd_avaliacoes_usuarios")).limit(10))

Filmes com avaliação mas fora do catálogo (descartados): 441
Unicidade OK  dim_reviews.sk_movie_id  (27,226 linhas)
OK  cinedata_analytics.gold.dim_reviews: 27,226 linhas


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
401004439408690949,3177317153214777,5,4.73
695710643565982927,626434812484582680,4,4.9
360829838431534330,114316997569944513,4,5.27
395881600248758867,397598355825751257,4,5.68
1107244855957133531,718697457622465184,4,3.93
336242642786879766,79164677404047279,4,6.23
1071465515828161279,369218759801783514,4,3.35
310123118839038626,1086601039921956800,4,4.38
30324924203125941,514076458638422835,4,7.4
512789610794696784,1040828299171320321,4,7.07


In [0]:
# =============================================================================
# 6. gold.fact_movies_performance
# =============================================================================
# GRÃO: 1 linha por filme LANÇADO.
# Como garantir que os joins não dupliquem o grão:
#  (a) A base da fato é a dim_movies filtrada ("Lançado"), com 1 linha por filme (unicidade já validada).
#  (b) Antes do join, validamos que financeiro e métricas também têm 1 linha por id_filme.
#      Um join 1:1 nunca multiplica linhas; bastaria uma chave repetida na Silver para duplicar.
#  (c) LEFT JOIN: filmes sem dados financeiros/engajamento continuam na fato (métricas NULL),
#      em vez de sumirem, como aconteceria num inner join.
#  (d) Validação final: nº de linhas da fato == nº de filmes lançados e sk_movie_id único.
#  (e) Gêneros, pessoas e produtoras NÃO entram na fato (são N:N); ficam nas bridge tables.
df_financeiro = ler_silver("tb_financeiro_filmes")
df_metricas   = ler_silver("tb_metricas_engajamento")
validar_unicidade(df_financeiro, "id_filme", "silver.tb_financeiro_filmes")
validar_unicidade(df_metricas,   "id_filme", "silver.tb_metricas_engajamento")

df_lancados = ler_gold("dim_movies").filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")

fact = (df_lancados
    .join(df_financeiro, "id_filme", "left")
    .join(df_metricas,   "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("BIGINT"),
        # Métricas financeiras
        F.col("orcamento_usd").cast("DECIMAL(18,2)"),
        F.col("receita_usd").cast("DECIMAL(18,2)"),
        F.col("lucro_usd").cast("DECIMAL(18,2)"),
        F.col("orcamento_brl").cast("DECIMAL(18,2)"),
        F.col("receita_brl").cast("DECIMAL(18,2)"),
        F.col("lucro_brl").cast("DECIMAL(18,2)"),
        # Métricas de engajamento
        F.col("popularidade").cast("DOUBLE"),
        F.col("nota_media_tmdb").cast("DOUBLE"),
        F.col("qtd_votos_tmdb").cast("INT"),
        F.col("nota_media_imdb").cast("DOUBLE"),
        F.col("qtd_votos_imdb").cast("INT"),
    ))

validar_unicidade(fact, "sk_movie_id", "fact_movies_performance")
qtd_lancados, qtd_fato = df_lancados.count(), fact.count()
assert qtd_fato == qtd_lancados, f"Grão violado: {qtd_fato} linhas na fato vs {qtd_lancados} filmes lançados"
print(f"Grão OK: {qtd_fato:,} linhas = {qtd_lancados:,} filmes lançados")

salvar_gold(fact, "fact_movies_performance")
display(ler_gold("fact_movies_performance").orderBy(F.desc("receita_usd")).limit(10))

Unicidade OK  silver.tb_financeiro_filmes.id_filme  (99,006 linhas)
Unicidade OK  silver.tb_metricas_engajamento.id_filme  (95,115 linhas)
Unicidade OK  fact_movies_performance.sk_movie_id  (96,261 linhas)
Grão OK: 96,261 linhas = 96,261 filmes lançados
OK  cinedata_analytics.gold.fact_movies_performance: 96,261 linhas


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
185237272942503179,356000000.00,2800000000.00,2444000000.00,1833934000.00,14424200000.00,12590266000.00,91.756,8.263,23857,8.4,1484150
842779332390686918,460000000.00,2320250281.00,1860250281.00,2369690000.00,11952769322.57,9583079322.57,241.285,null,9830,7.5,646896
433553154783147502,300000000.00,2052415039.00,1752415039.00,1545450000.00,10573016073.41,9027566073.41,154.34,8.255,27713,8.4,1406782
180778299061029074,200000000.00,1921847111.00,1721847111.00,1030300000.00,9900395392.32,8870095392.32,186.065,7.99,18299,null,1070613
960698904292846382,260000000.00,1663075401.00,1403075401.00,1339390000.00,8567332928.25,7227942928.25,63.351,7.1,null,6.8,295367
143165421211047770,170000000.00,1488732821.00,1318732821.00,875755000.00,7669207127.38,6793452127.38,126.291,8.26,7546,8.2,903484
64783273453943383,145000000.00,1428545028.00,1283545028.00,746967500.00,7359149711.74,6612182211.74,1069.34,7.279,5074,6.8,704472
825613619021979764,100000000.00,1355725263.00,1255725263.00,515150000.00,6984018692.34,6468868692.34,410.411,7.775,6743,7.0,301951
1058276972530113905,200000000.00,1349926083.00,1149926083.00,1030300000.00,6954144216.57,5923844216.57,43.665,7.39,null,7.3,924922
585007014835674829,200000000.00,1332698830.00,1132698830.00,1030300000.00,6865398022.75,5835098022.75,47.241,6.825,14312,6.8,726501


In [0]:
# =============================================================================
# 7. Bridge tables (relações N:N)
# =============================================================================
# Um filme tem vários gêneros/pessoas/produtoras e vice-versa. Se esses atributos fossem
# levados para a fato, cada filme se repetiria N vezes (ex.: 3 gêneros × 10 atores = 30 linhas),
# e um SUM(receita) contaria a mesma receita 30 vezes.
# A bridge guarda só os pares (sk_movie_id, sk_dimensão): a fato continua com 1 linha por filme,
# e a ligação com a dimensão periférica é feita pela bridge apenas quando a análise precisar.
#
# Construção: Silver (id_filme + chave natural) → join dim_movies (sk_movie_id)
#             → join dimensão periférica (sk_*) → distinct (evita pares repetidos).
sk_filmes = ler_gold("dim_movies").select("id_filme", "sk_movie_id")
pessoas_empresas = ler_silver("tb_pessoas_empresas")

bridge_movie_genre = (ler_silver("tb_generos")
    .join(sk_filmes, "id_filme")
    .join(ler_gold("dim_genres"), "nome_genero")
    .select("sk_movie_id", "sk_genre_id").distinct())

bridge_movie_person = (pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .join(sk_filmes, "id_filme")
    .join(ler_gold("dim_people"), ["nome_pessoa", "tipo_pessoa"])
    .select("sk_movie_id", "sk_person_id").distinct())

bridge_movie_company = (pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .join(sk_filmes, "id_filme")
    .join(ler_gold("dim_companies"), "nome_produtora")
    .select("sk_movie_id", "sk_company_id").distinct())

salvar_gold(bridge_movie_genre,   "bridge_movie_genre")
salvar_gold(bridge_movie_person,  "bridge_movie_person")
salvar_gold(bridge_movie_company, "bridge_movie_company")

display(ler_gold("bridge_movie_genre").limit(5))
display(ler_gold("bridge_movie_person").limit(5))
display(ler_gold("bridge_movie_company").limit(5))

OK  cinedata_analytics.gold.bridge_movie_genre: 139,862 linhas
OK  cinedata_analytics.gold.bridge_movie_person: 754,240 linhas
OK  cinedata_analytics.gold.bridge_movie_company: 116,470 linhas


sk_movie_id,sk_genre_id
141984717222225949,154830817088025809
1070355541195875344,983803378021321891
602107906575667249,269148797051626769
649981764307885307,269148797051626769
890290481946734251,663722414151260866


sk_movie_id,sk_person_id
26989153095801767,1051526310695410002
29877186394464393,803564383020973872
741472004781485382,630202362482825497
1138358749585544004,778816909336167591
188323232998749855,595113142302300265


sk_movie_id,sk_company_id
259732003697216809,250674605231101122
1059883003639732901,1101087988171892848
63757909455461461,744829222414909933
610730936237239635,980895148853657069
654743461809980187,1024939821094334428


In [0]:
# =============================================================================
# 8. Validação do Star Schema: integridade referencial + prova de que a fato não explode
# =============================================================================
# Integridade: toda FK precisa existir na dimensão correspondente (esperado: 0 órfãos).
checagens = [
    ("fact_movies_performance", "sk_movie_id",   "dim_movies",    "sk_movie_id"),
    ("dim_reviews",             "sk_movie_id",   "dim_movies",    "sk_movie_id"),
    ("bridge_movie_genre",      "sk_movie_id",   "dim_movies",    "sk_movie_id"),
    ("bridge_movie_genre",      "sk_genre_id",   "dim_genres",    "sk_genre_id"),
    ("bridge_movie_person",     "sk_movie_id",   "dim_movies",    "sk_movie_id"),
    ("bridge_movie_person",     "sk_person_id",  "dim_people",    "sk_person_id"),
    ("bridge_movie_company",    "sk_movie_id",   "dim_movies",    "sk_movie_id"),
    ("bridge_movie_company",    "sk_company_id", "dim_companies", "sk_company_id"),
]
resultado = []
for origem, fk, dimensao, pk in checagens:
    orfaos = (ler_gold(origem).select(F.col(fk).alias("k"))
              .join(ler_gold(dimensao).select(F.col(pk).alias("k")), "k", "left_anti").count())
    resultado.append((origem, fk, dimensao, orfaos))
display(spark.createDataFrame(resultado, ["tabela", "fk", "dimensao", "registros_orfaos"]))

# Demonstração de por que a bridge existe: a receita total pela fato (1 linha por filme) é o valor
# correto. Se a fato for juntada direto com a bridge de gêneros e somada, cada filme é contado
# uma vez por gênero, e o total fica inflado. Por isso, métricas se somam na fato, e a bridge
# serve para filtrar/agrupar por dimensão (ex.: COUNT DISTINCT de filmes por gênero).
fato = ler_gold("fact_movies_performance")
receita_correta = fato.agg(F.sum("receita_usd")).first()[0]
receita_errada  = (fato.join(ler_gold("bridge_movie_genre"), "sk_movie_id")
                       .agg(F.sum("receita_usd")).first()[0])
print(f"Receita total (fato, 1 linha por filme):          US$ {receita_correta:,.2f}")
print(f"Receita somada após join ingênuo com gêneros:     US$ {receita_errada:,.2f}  ← contada várias vezes")

tabela,fk,dimensao,registros_orfaos
fact_movies_performance,sk_movie_id,dim_movies,0
dim_reviews,sk_movie_id,dim_movies,0
bridge_movie_genre,sk_movie_id,dim_movies,0
bridge_movie_genre,sk_genre_id,dim_genres,0
bridge_movie_person,sk_movie_id,dim_movies,0
bridge_movie_person,sk_person_id,dim_people,0
bridge_movie_company,sk_movie_id,dim_movies,0
bridge_movie_company,sk_company_id,dim_companies,0


Receita total (fato, 1 linha por filme):          US$ 162,307,288,955.00
Receita somada após join ingênuo com gêneros:     US$ 484,261,743,659.00  ← contada várias vezes


In [0]:
# =============================================================================
# Resumo da camada Gold (Star Schema)
# =============================================================================
tabelas_gold = ["fact_movies_performance", "dim_movies", "dim_genres", "dim_people", "dim_companies",
                "dim_reviews", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company"]
display(spark.createDataFrame([(t, ler_gold(t).count()) for t in tabelas_gold], ["tabela", "linhas"]))

tabela,linhas
fact_movies_performance,96261
dim_movies,97611
dim_genres,19
dim_people,417525
dim_companies,45741
dim_reviews,27226
bridge_movie_genre,139862
bridge_movie_person,754240
bridge_movie_company,116470


In [0]:
# =============================================================================
# 9. gold.gold_genai_movies_context  (tabela de contexto para o RAG / Vector Search)
# =============================================================================
# Problema: concat() e || retornam NULL se QUALQUER campo for NULL, e o filme inteiro some.
# Solução: coalesce() em CADA campo ANTES da concatenação, com um texto de fallback.
#
# Análise de nulos (quais campos podem faltar e qual fallback faz sentido):
#   título     → raro, mas possível            → "Título não informado"
#   ano        → 2 filmes sem data             → "ano não informado"
#   receita    → ~96% dos filmes (receita 0/Unknown na origem, e filmes não lançados não estão na fato)
#                                              → "valor não divulgado"
#   orçamento  → ~92% dos filmes               → "valor não divulgado"
#   atores     → filmes sem elenco (documentários, curtas) → "elenco não informado"
#   diretor    → filmes sem diretor na origem  → "diretor não informado"
#   sinopse    → NULL ou vazia                 → "Sinopse não disponível."
# O fallback mantém a frase legível e deixa claro para o LLM que o dado NÃO EXISTE (em vez de
# ser zero): ele não inventa "faturou US$ 0" e o filme não some da base vetorial.
from pyspark.sql.window import Window

def formatar_usd(coluna):
    """US$ com separador de milhar brasileiro: 300000000.00 → 'US$ 300.000.000'.
    format_number gera '300,000,000' (padrão americano); translate troca ',' por '.'.
    NULL continua NULL, e o fallback é aplicado depois, na montagem do texto."""
    return F.concat(F.lit("US$ "), F.translate(F.format_number(F.col(coluna), 0), ",", "."))

def juntar_nomes(coluna_array):
    """Transforma um array em texto natural: [A] → 'A' | [A,B] → 'A e B' | [A,B,C] → 'A, B e C'.
    Array vazio ou NULL → NULL (o fallback é aplicado na montagem)."""
    arr, n = F.col(coluna_array), F.size(F.col(coluna_array))
    return (F.when(arr.isNull() | (n <= 0), F.lit(None))
             .when(n == 1, F.element_at(arr, 1))
             .otherwise(F.concat(F.array_join(F.slice(arr, 1, n - 1), ", "),
                                 F.lit(" e "), F.element_at(arr, -1))))

In [0]:
# -----------------------------------------------------------------------------
# Atores principais e diretores por filme (via bridge_movie_person + dim_people)
# -----------------------------------------------------------------------------
# A ordem de crédito (billing) não existe no modelo: a bridge guarda só pares (filme, pessoa).
# Critério para os "atores principais": os 5 atores com MAIS filmes no catálogo, como proxy de
# relevância/notoriedade, com desempate alfabético. Isso deixa a escolha determinística
# (collect_list sozinho devolveria uma ordem aleatória a cada execução).
QTD_ATORES_PRINCIPAIS = 5

pessoas_filme = (ler_gold("bridge_movie_person")
    .join(ler_gold("dim_people"), "sk_person_id"))

atores = (pessoas_filme.filter(F.col("tipo_pessoa") == "Ator")
    .withColumn("qtd_filmes", F.count("*").over(Window.partitionBy("sk_person_id")))
    .groupBy("sk_movie_id")
    # sort_array de struct(-qtd, nome): ordena por mais filmes e depois pelo nome
    .agg(F.sort_array(F.collect_list(F.struct((-F.col("qtd_filmes")).alias("ordem"), "nome_pessoa"))).alias("_lista"))
    .select("sk_movie_id",
            F.slice(F.col("_lista.nome_pessoa"), 1, QTD_ATORES_PRINCIPAIS).alias("_atores")))

diretores = (pessoas_filme.filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.sort_array(F.collect_set("nome_pessoa")).alias("_diretores")))   # todos, em ordem alfabética

In [0]:
# -----------------------------------------------------------------------------
# Montagem do documento (null-safe)
# -----------------------------------------------------------------------------
# Base = dim_movies (TODOS os filmes do catálogo) + LEFT JOINs: nenhum filme é perdido nos joins.
# Filmes não lançados não estão na fato, então receita/orçamento caem no fallback.
base = (ler_gold("dim_movies")
    .join(ler_gold("fact_movies_performance").select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
    .join(atores,    "sk_movie_id", "left")
    .join(diretores, "sk_movie_id", "left"))

# Sinopse: vazia/só espaços → NULL (nullif), para cair no fallback. O template termina com ".",
# então só adicionamos o ponto final se a sinopse ainda não terminar em pontuação (evita "..").
sinopse = F.expr("nullif(trim(sinopse), '')")
sinopse = F.when(sinopse.rlike(r"[.!?…]$"), sinopse).otherwise(F.concat(sinopse, F.lit(".")))

# coalesce em CADA campo ANTES da concatenação: nenhum valor NULL chega ao concat()
titulo    = F.coalesce(F.expr("nullif(trim(titulo), '')"), F.lit("Título não informado"))
ano       = F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado"))
receita   = F.coalesce(formatar_usd("receita_usd"),   F.lit("valor não divulgado"))
orcamento = F.coalesce(formatar_usd("orcamento_usd"), F.lit("valor não divulgado"))
elenco    = F.coalesce(juntar_nomes("_atores"),       F.lit("elenco não informado"))
direcao   = F.coalesce(juntar_nomes("_diretores"),    F.lit("diretor não informado"))
sinopse   = F.coalesce(sinopse,                       F.lit("Sinopse não disponível."))

documento = F.concat(
    F.lit("O filme "), titulo,
    F.lit(", lançado no ano de "), ano,
    F.lit(", faturou "), receita,
    F.lit(" e teve um custo de "), orcamento,
    F.lit(". Estrelado por "), elenco,
    F.lit(" e dirigido por "), direcao,
    F.lit(", o filme possui a seguinte sinopse: "), sinopse,
)

gold_genai = base.select(
    F.col("id_filme").cast("STRING").alias("movie_id"),
    titulo.alias("title"),
    documento.cast("STRING").alias("llm_context_document"),
)

In [0]:
# -----------------------------------------------------------------------------
# Validação ANTES de gravar: prova de que nenhum filme desapareceu
# -----------------------------------------------------------------------------
qtd_filmes     = ler_gold("dim_movies").count()
qtd_documentos = gold_genai.count()
qtd_nulos      = gold_genai.filter(F.col("llm_context_document").isNull()).count()

assert qtd_documentos == qtd_filmes, f"Filmes perdidos: {qtd_filmes - qtd_documentos}"
assert qtd_nulos == 0, f"{qtd_nulos} documentos NULL"
validar_unicidade(gold_genai, "movie_id", "gold_genai_movies_context")
print(f"OK: {qtd_documentos:,} documentos para {qtd_filmes:,} filmes, 0 nulos")

salvar_gold(gold_genai, "gold_genai_movies_context")

# Change Data Feed: requisito para criar um índice Delta Sync no Databricks Vector Search,
# que sincroniza o índice incrementalmente quando a tabela muda.
spark.sql(f"ALTER TABLE {GOLD}.gold_genai_movies_context SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

Unicidade OK  gold_genai_movies_context.movie_id  (97,611 linhas)
OK: 97,611 documentos para 97,611 filmes, 0 nulos
OK  cinedata_analytics.gold.gold_genai_movies_context: 97,611 linhas


DataFrame[]

In [0]:
# -----------------------------------------------------------------------------
# Inspeção dos documentos gerados
# -----------------------------------------------------------------------------
ctx = ler_gold("gold_genai_movies_context")

# Filmes com todos os dados preenchidos
display(ctx.filter(~F.col("llm_context_document").contains("não divulgado")
                   & ~F.col("llm_context_document").contains("não informado")).limit(5))

# Quantos filmes usaram cada fallback (antes, eles teriam sumido da tabela)
display(ctx.agg(
    F.count("*").alias("documentos"),
    F.sum(F.col("llm_context_document").contains("faturou valor não divulgado").cast("int")).alias("sem_receita"),
    F.sum(F.col("llm_context_document").contains("custo de valor não divulgado").cast("int")).alias("sem_orcamento"),
    F.sum(F.col("llm_context_document").contains("elenco não informado").cast("int")).alias("sem_elenco"),
    F.sum(F.col("llm_context_document").contains("diretor não informado").cast("int")).alias("sem_diretor"),
    F.sum(F.col("llm_context_document").contains("Sinopse não disponível").cast("int")).alias("sem_sinopse"),
))

# Exemplos de filmes que só existem na tabela graças ao fallback
display(ctx.filter(F.col("llm_context_document").contains("diretor não informado")).limit(5))

movie_id,title,llm_context_document
1001724,Drummies,"O filme Drummies, lançado no ano de 2022, faturou US$ 8.000 e teve um custo de US$ 3.000. Estrelado por Danique Africa e dirigido por Jessie Zinn, o filme possui a seguinte sinopse: In a meditation on the meaning behind sports in a Post-Apartheid South Africa, three young girls muse on their hopes and dreams as aspiring Drum Majorettes."
1024777,Bai Ivan 2,"O filme Bai Ivan 2, lançado no ano de 2022, faturou US$ 60.000 e teve um custo de US$ 10.000. Estrelado por Dimitar Kiriazov, Silvia Yordanova e Nikolai Kokurinkov e dirigido por Nikolai Garabedian, o filme possui a seguinte sinopse: \A sequel to the Bulgarian 2021 movie \""\""Bai Ivan\""\""\""""."
1032048,Priyotoma,"O filme Priyotoma, lançado no ano de 2023, faturou US$ 3.800.000 e teve um custo de US$ 230.000. Estrelado por Shakib Khan, Shahiduzzaman Selim, Lutfur Rahman George, Shiba Shanu e Elina Shammi e dirigido por Himel Ashraf, o filme possui a seguinte sinopse: Sinopse não disponível."
1040266,Time Trap,"O filme Time Trap, lançado no ano de 2021, faturou US$ 1.000 e teve um custo de US$ 100. Estrelado por Catesharon Gakeni, Bill Clinton Isaac, James Caesar, Rutherford Moemga e Samwel Bogonko e dirigido por Allan Bosire, o filme possui a seguinte sinopse: A Time Traveller who is Trapped Back in Time and In The Process Comes Across Native Residents Whom She Has To Encounter With So She Can Travel Back To Her Timeline."
1048522,Fremont,"O filme Fremont, lançado no ano de 2023, faturou US$ 281.126 e teve um custo de US$ 3.000.000. Estrelado por Jeremy Allen White, Gregg Turkington, Anaita Wali Zada, Avis See-tho e Divya Jakatdar e dirigido por Babak Jalali, o filme possui a seguinte sinopse: Donya, a lonely Afghan refugee and former translator, spends her twenties drifting through a meager existence in Fremont, California. Shuttling between her job writing fortunes for a fortune cookie factory and sessions with her eccentric therapist, Donya suffers from insomnia and survivor's guilt over those still left behind in Kabul as she desperately searches for love."


documentos,sem_receita,sem_orcamento,sem_elenco,sem_diretor,sem_sinopse
97611,94345,89646,17951,18018,13228


movie_id,title,llm_context_document
1000091,Amy Miller: Ham Mouth,"O filme Amy Miller: Ham Mouth, lançado no ano de 2022, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por Amy C. Miller e dirigido por diretor não informado, o filme possui a seguinte sinopse: Amy Miller reflects on her recent breakup, shares her love of baths and reveals the 40-year-old shit she does."
1000116,Don't Wait for Me,"O filme Don't Wait for Me, lançado no ano de 2021, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por Uri Gavriel, Einat Saruf, Israel Ogalbo, Omer Hazan e Naor Vaturi e dirigido por diretor não informado, o filme possui a seguinte sinopse: Matan arrives in a crime neighborhood after his father goes bankrupt and loses their home."
1000197,The Hidden Fox,"O filme The Hidden Fox, lançado no ano de 2022, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por Ray Lui, Yang Yi, Chen Yusi, Chunyu Shanshan e Ben Ng Ngai-Cheung e dirigido por diretor não informado, o filme possui a seguinte sinopse: Ten years ago, in order to snatch the mysterious treasure left by Dashing King Li Zicheng, the wicked designed a decisive battle between the hero Miao Renfeng and the treasure guardian Hu. Hu and Miao died tragically, but the treasure map disappeared. In ten years, they have never given up the search for treasure. Finally, the treasure map reappeared, and the eight evil men gathered together and set off to Feihu Mountain.. (Source: forum.soompi)."
1000362,Jim Breuer: Somebody Had to Say It,"O filme Jim Breuer: Somebody Had to Say It, lançado no ano de 2021, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por Jim Breuer e dirigido por diretor não informado, o filme possui a seguinte sinopse: New Standup comedy special from Jim Breuer."
1000409,Strangers by Night,"O filme Strangers by Night, lançado no ano de 2023, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por elenco não informado e dirigido por diretor não informado, o filme possui a seguinte sinopse: \In a crowded subway train."


In [0]:
# =============================================================================
# ANALYTICS — Pergunta 1: Receita total (R$) de todos os filmes
# =============================================================================
# - SUM ignora NULLs: filmes sem receita conhecida não entram na soma (em vez de contarem como 0,
#   o resultado é o mesmo, mas deixamos explícito quantos filmes compõem o total).
# - A soma é feita na FATO (1 linha por filme). Se fosse feita após um join com qualquer bridge,
#   a receita seria contada várias vezes (ver validação do Star Schema).
display(spark.sql(f"""
    SELECT
        SUM(receita_brl)            AS receita_total_brl,
        SUM(receita_usd)            AS receita_total_usd,
        COUNT(receita_brl)          AS filmes_com_receita,
        COUNT(*)                    AS filmes_na_base
    FROM {GOLD}.fact_movies_performance
"""))

receita_total_brl,receita_total_usd,filmes_com_receita,filmes_na_base
836125999052.20,162307288955.00,3266,96261


In [0]:
# =============================================================================
# Pergunta 2: Top 5 filmes por popularidade
# =============================================================================
# - Fato (popularidade) + dim_movies (título) pela surrogate key.
# - Filmes com popularidade NULL ficam fora do ranking.
display(spark.sql(f"""
    SELECT
        m.titulo,
        f.popularidade
    FROM {GOLD}.fact_movies_performance f
    JOIN {GOLD}.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
Blue Beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
Retribution,1547.22


In [0]:
# =============================================================================
# Pergunta 3: Quantidade de filmes por gênero
# =============================================================================
# - Relação N:N → passa pela bridge_movie_genre.
# - COUNT(DISTINCT sk_movie_id): conta FILMES, não linhas; protege contra qualquer par repetido.
# - Um filme com 3 gêneros conta 1 vez em cada um dos 3 gêneros (é o esperado nesse tipo de pergunta),
#   por isso a soma da coluna é maior que o total de filmes.
# - Considera o catálogo completo (dim_movies), lançados ou não.
display(spark.sql(f"""
    SELECT
        g.nome_genero,
        COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM {GOLD}.bridge_movie_genre b
    JOIN {GOLD}.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


In [0]:
# =============================================================================
# Pergunta 4: Top 10 filmes por receita com RANK()
# =============================================================================
# Window Function: RANK() OVER (ORDER BY receita_usd DESC)
# - Sem PARTITION BY: o ranking é global (todos os filmes competem entre si).
# - RANK dá a MESMA posição para empates e pula as seguintes (1, 2, 2, 4...).
#   Por isso filtramos por posição (<= 10) e não com LIMIT 10: se houver empate na 10ª posição,
#   todos os empatados aparecem, o que é o resultado justo.
# - O ranking é calculado numa subquery (CTE) porque não dá para filtrar uma window function
#   no mesmo nível em que ela é calculada (o WHERE roda antes do SELECT).
# - Ordenamos por USD (moeda original). A ordem em BRL é idêntica, pois a cotação é a mesma para todos.
display(spark.sql(f"""
    WITH ranking AS (
        SELECT
            m.titulo,
            f.receita_usd,
            f.receita_brl,
            RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM {GOLD}.fact_movies_performance f
        JOIN {GOLD}.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT posicao_ranking, titulo, receita_usd, receita_brl
    FROM ranking
    WHERE posicao_ranking <= 10
    ORDER BY posicao_ranking, titulo
"""))

posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14424200000.00
2,Avatar: The Way of Water,2320250281.00,11952769322.57
3,Avengers: Infinity War,2052415039.00,10573016073.41
4,Spider-Man: No Way Home,1921847111.00,9900395392.32
5,The Lion King,1663075401.00,8567332928.25
6,Top Gun: Maverick,1488732821.00,7669207127.38
7,Barbie,1428545028.00,7359149711.74
8,The Super Mario Bros. Movie,1355725263.00,6984018692.34
9,Black Panther,1349926083.00,6954144216.57
10,Star Wars: The Last Jedi,1332698830.00,6865398022.75


In [0]:
# =============================================================================
# Data de referência para as perguntas 5 e 6
# =============================================================================
# Regra do escopo: o limite superior é a data de lançamento REALIZADA mais recente da base,
# ignorando filmes não lançados e datas futuras. Não usamos current_date() como referência:
# a base é um retrato histórico, e "últimos 2 anos" é relativo aos dados, não ao dia de hoje.
#   - status_filme = 'Lançado'      → exclui Planejado, Em Produção, Pós-Produção, Rumores...
#   - data_lancamento <= hoje       → exclui datas futuras (erro de cadastro ou pré-estreia)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_data_referencia AS
    SELECT MAX(data_lancamento) AS data_ref
    FROM {GOLD}.dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento <= current_date()
""")
display(spark.sql("""
    SELECT data_ref,
           add_months(data_ref, -24) AS inicio_janela_2_anos,
           add_months(data_ref, -60) AS inicio_janela_5_anos
    FROM vw_data_referencia
"""))

data_ref,inicio_janela_2_anos,inicio_janela_5_anos
2026-02-19,2024-02-19,2021-02-19


In [0]:
# =============================================================================
# Pergunta 5: Ator com mais participações em filmes lançados nos últimos 2 anos
# =============================================================================
# - Janela: data_lancamento entre (data_ref - 24 meses) e data_ref. add_months respeita
#   o calendário (anos bissextos, meses de tamanhos diferentes), ao contrário de "- 730 dias".
# - Somente filmes lançados (mesmo critério da data de referência).
# - COUNT(DISTINCT chave_obra): cada FILME conta uma vez por ator. A base tem o mesmo filme cadastrado
#   com ids diferentes (223 filmes, 411 ids excedentes, ex.: 'Die Hart 2: Die Harter' com 25 ids).
#   Contar por id faria o elenco desse filme "participar" de 25 filmes. A obra é identificada por
#   título normalizado + data de lançamento.
# - RANK() em vez de LIMIT 1: se dois atores empatarem no topo, ambos aparecem.
display(spark.sql(f"""
    WITH filmes_janela AS (
        SELECT m.sk_movie_id,
               -- "Obra": título normalizado + data (o mesmo filme pode ter vários ids)
               concat(lower(trim(m.titulo)), '|', m.data_lancamento) AS chave_obra
        FROM {GOLD}.dim_movies m
        CROSS JOIN vw_data_referencia r
        WHERE m.status_filme = 'Lançado'
          AND m.data_lancamento BETWEEN add_months(r.data_ref, -24) AND r.data_ref
    ),
    participacoes AS (
        SELECT
            p.nome_pessoa                  AS ator,
            COUNT(DISTINCT fj.chave_obra)  AS qtd_participacoes
        FROM filmes_janela fj
        JOIN {GOLD}.bridge_movie_person b ON b.sk_movie_id  = fj.sk_movie_id
        JOIN {GOLD}.dim_people p          ON p.sk_person_id = b.sk_person_id
        WHERE p.tipo_pessoa = 'Ator'
        GROUP BY p.nome_pessoa
    ),
    ranking AS (
        SELECT *, RANK() OVER (ORDER BY qtd_participacoes DESC) AS posicao
        FROM participacoes
    )
    SELECT posicao, ator, qtd_participacoes
    FROM ranking
    WHERE posicao <= 5            -- top 5 para contexto; a resposta é a posição 1
    ORDER BY posicao, ator
"""))

posicao,ator,qtd_participacoes
1,Kevin Hart,13
2,Alex Dauphin,12
2,Alon McKlveen,12
2,Ben Schwartz,12
2,Branden Morgan,12
2,Brandon H. Morgan,12
2,Brandon Morgan,12
2,David Chan,12
2,David Chang,12
2,Forrest Conoly,12


In [0]:
# =============================================================================
# Pergunta 6: Produtora com maior lucro nos últimos 5 anos
# =============================================================================
# - Janela: (data_ref - 60 meses) até data_ref, somente filmes lançados.
# - Lucro vem da FATO (receita - orçamento). Filmes sem lucro calculável (orçamento ou receita
#   ausentes) ficam de fora: tratar como 0 distorceria o ranking.
# - Mesmo filme com ids diferentes: agregamos por obra (título + data) antes de somar, para não
#   contar o lucro das cópias duplicadas.
# - Coprodução: um filme com 2 produtoras soma seu lucro integral para CADA uma delas.
#   Isso é correto para a pergunta "quanto lucro os filmes de cada produtora geraram", mas por
#   isso a soma da coluna não é o lucro total do mercado.
# - Mostramos USD e BRL; o ranking é por USD (moeda original).
display(spark.sql(f"""
    WITH filmes_janela AS (
        SELECT f.sk_movie_id, f.lucro_usd, f.lucro_brl,
               concat(lower(trim(m.titulo)), '|', m.data_lancamento) AS chave_obra
        FROM {GOLD}.fact_movies_performance f
        JOIN {GOLD}.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        CROSS JOIN vw_data_referencia r
        WHERE m.status_filme = 'Lançado'
          AND m.data_lancamento BETWEEN add_months(r.data_ref, -60) AND r.data_ref
          AND f.lucro_usd IS NOT NULL
    ),
    -- 1 linha por (produtora, obra): cópias do mesmo filme com ids diferentes não somam lucro 2x
    lucro_obra AS (
        SELECT c.nome_produtora, fj.chave_obra,
               MAX(fj.lucro_usd) AS lucro_usd, MAX(fj.lucro_brl) AS lucro_brl
        FROM filmes_janela fj
        JOIN {GOLD}.bridge_movie_company b ON b.sk_movie_id   = fj.sk_movie_id
        JOIN {GOLD}.dim_companies c        ON c.sk_company_id = b.sk_company_id
        GROUP BY c.nome_produtora, fj.chave_obra
    ),
    lucro_produtora AS (
        SELECT
            nome_produtora,
            SUM(lucro_usd)  AS lucro_total_usd,
            SUM(lucro_brl)  AS lucro_total_brl,
            COUNT(*)        AS qtd_filmes_com_lucro_apurado
        FROM lucro_obra
        GROUP BY nome_produtora
    ),
    ranking AS (
        SELECT *, RANK() OVER (ORDER BY lucro_total_usd DESC) AS posicao
        FROM lucro_produtora
    )
    SELECT posicao, nome_produtora, lucro_total_usd, lucro_total_brl, qtd_filmes_com_lucro_apurado
    FROM ranking
    WHERE posicao <= 5            -- top 5 para contexto; a resposta é a posição 1
    ORDER BY posicao
"""))

posicao,nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes_com_lucro_apurado
1,Universal Pictures,5772329679.00,29736156341.39,24
2,Marvel Studios,4953462823.00,25517763732.69,8
3,Columbia Pictures,3662050755.00,18865054464.39,12
4,Pascal Pictures,2701952454.00,13919108066.79,3
5,Illumination,2431353473.00,12525117416.16,3
